# Partie 2.2 – Analyses exploratoires descriptives

In [ ]:
%%sql -r dataframe_2
use database anycompany_lab;
use schema silver;

# Analyse de l’évolution des ventes dans le temps

## Ventes par mois

In [ ]:
%%sql -r dataframe_1
SELECT
    DATE_TRUNC('MONTH', transaction_date) AS month,
    SUM(amount) AS total_sales,
    COUNT(*) AS number_of_sales
FROM financial_transactions_clean
WHERE transaction_type = 'Sale'
GROUP BY month
ORDER BY month;

## Ventes par année

In [ ]:
%%sql -r dataframe_3
SELECT
    YEAR(transaction_date) AS year,
    SUM(amount) AS total_sales
FROM financial_transactions_clean
WHERE transaction_type = 'Sale'
GROUP BY year
ORDER BY year;

## Évolution cumulée des ventes

### évolution cumulée par jour de vente

In [ ]:
%%sql -r dataframe_4
SELECT
    transaction_date,
    SUM(amount) OVER (ORDER BY transaction_date) AS cumulative_sales
FROM financial_transactions_clean
WHERE transaction_type = 'Sale'
ORDER BY transaction_date;

### évolution cumulée par mois

In [ ]:
%%sql -r dataframe_14
WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('MONTH', transaction_date) AS sales_month,
        SUM(amount) AS monthly_amount
    FROM SILVER.FINANCIAL_TRANSACTIONS_CLEAN
    WHERE transaction_type = 'Sale'
    GROUP BY 1
)
SELECT
    sales_month,
    monthly_amount,
    SUM(monthly_amount) OVER (ORDER BY sales_month) AS cumulative_sales_per_month
FROM monthly_sales
ORDER BY sales_month;

### évolution cumulée par an

In [ ]:
%%sql -r dataframe_15
WITH yearly_sales AS (
    SELECT
        DATE_TRUNC('YEAR', transaction_date) AS sales_year,
        SUM(amount) AS yearly_amount
    FROM SILVER.FINANCIAL_TRANSACTIONS_CLEAN
    WHERE transaction_type = 'Sale'
    GROUP BY 1
)
SELECT
    sales_year,
    yearly_amount,
    SUM(yearly_amount) OVER (ORDER BY sales_year) AS cumulative_sales_per_year
FROM yearly_sales
ORDER BY sales_year;

# Performance par produit, catégorie et région

## Performance par région

In [ ]:
%%sql -r dataframe_5
SELECT
    region,
    SUM(amount) AS total_sales,
    COUNT(*) AS total_transactions,
    ROUND(AVG(amount), 2) AS avg_transaction
FROM SILVER.FINANCIAL_TRANSACTIONS_CLEAN
WHERE transaction_type = 'Sale'
GROUP BY region
ORDER BY total_sales DESC;

## Performance par produit

In [ ]:
%%sql -r dataframe_18
SELECT 
    p.product_category,
    COUNT(ft.transaction_id) AS nb_ventes,
    SUM(ft.amount) AS total_revenue,
    ROUND(AVG(ft.amount),2) AS avg_order_value
FROM SILVER.FINANCIAL_TRANSACTIONS_CLEAN ft
LEFT JOIN SILVER.PROMOTIONS_DATA_CLEAN p
    ON ft.region = p.region
WHERE ft.transaction_type = 'Sale'
GROUP BY p.product_category;

## Performance par catégorie produit

In [ ]:
%%sql -r dataframe_6
SELECT
    product_category,
    ROUND(AVG(rating),1) AS avg_rating
FROM product_reviews_clean 
GROUP BY product_category
ORDER BY avg_rating DESC;

## Performance promotions par région

In [ ]:
%%sql -r dataframe_7
SELECT
    region,
    ROUND(AVG(discount_percentage),2) AS avg_discount,
    COUNT(*) AS number_of_promotions
FROM promotions_data_clean
GROUP BY region
ORDER BY avg_discount DESC;

## Performance campagnes marketing

In [ ]:
%%sql -r dataframe_8
SELECT
    campaign_type,
    SUM(budget) AS total_budget,
    SUM(reach) AS total_reach,
    AVG(conversion_rate) AS avg_conversion
FROM marketing_campaigns_clean
GROUP BY campaign_type
ORDER BY total_budget DESC;

# Répartition des clients par segments démographiques

## Clients par genre

In [ ]:
%%sql -r dataframe_9
SELECT
    gender,
    COUNT(*) AS number_of_customers
FROM customer_demographics_clean
GROUP BY gender;

## Clients par région

In [ ]:
%%sql -r dataframe_10
SELECT
    region,
    COUNT(*) AS customers
FROM customer_demographics_clean
GROUP BY region
ORDER BY customers DESC;

## Clients par tranche d’âge

In [ ]:
%%sql -r dataframe_11
SELECT
    CASE
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE) < 25 THEN 'Under 25'
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE) BETWEEN 25 AND 40 THEN '25-40'
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE) BETWEEN 41 AND 60 THEN '41-60'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS customers
FROM customer_demographics_clean
GROUP BY age_group
ORDER BY age_group;

## Clients par revenu

In [ ]:
%%sql -r dataframe_12
SELECT
    CASE
        WHEN annual_income < 30000 THEN 'Low Income'
        WHEN annual_income BETWEEN 30000 AND 70000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    COUNT(*) AS customers
FROM customer_demographics_clean
GROUP BY income_group;

## Client par revenu et par tranche d'âge

In [ ]:
%%sql -r dataframe_17
SELECT
    CASE
        WHEN annual_income < 30000 THEN 'Low Income'
        WHEN annual_income BETWEEN 30000 AND 70000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    
    CASE
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE()) < 25 THEN 'Under 25'
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE()) BETWEEN 25 AND 40 THEN '25-40'
        WHEN DATEDIFF(YEAR, date_of_birth, CURRENT_DATE()) BETWEEN 41 AND 60 THEN '41-60'
        ELSE '60+'
    END AS age_group,
    
    COUNT(*) AS customer_count
FROM SILVER.CUSTOMER_DEMOGRAPHICS_CLEAN
GROUP BY income_group, age_group
ORDER BY income_group, age_group;

Pour aller plus loin ...

In [ ]:
%%sql -r dataframe_13
SELECT
    m.campaign_id,
    m.region,
    SUM(f.amount) AS sales_after_campaign
FROM marketing_campaigns_clean m
JOIN financial_transactions_clean f
    ON m.region = f.region
    AND f.transaction_date BETWEEN m.start_date AND m.end_date
WHERE f.transaction_type = 'Sale'
GROUP BY m.campaign_name, m.region
ORDER BY sales_after_campaign DESC;